# Notebook 3: Related Work and Model Selection

This notebook covers **Section 3** of the assignment.

**Purpose:** Use the findings from EDA (Notebook 02) and the literature to
justify the three models selected for comparison.

**Research question:** Which three architecturally distinct sequential models
are best suited for one-step-ahead mobile network traffic forecasting on the
Milan dataset, and why?

---

## Summary of EDA Findings Relevant to Model Selection

Before selecting models, recall what the EDA revealed:

| Property | Finding | Modelling implication |
|---|---|---|
| Stationarity | ADF test: stationary (fill in p-value after running NB02) | Standard normalisation sufficient; no differencing required |
| Daily seasonality | Strong 24-hour cycle in ACF (peak at lag 144) | Model must have receptive field ≥ 144 steps |
| Weekly seasonality | Weaker but visible 7-day cycle (lag 1008) | Relevant for lookback choice; long-range dependency |
| Autocorrelation structure | Significant ACF up to ~2 days, rapid PACF cut-off after 1–3 lags | First-order dependencies dominate; but seasonal lags matter |
| Traffic distribution | Highly right-skewed across areas | High-traffic areas will dominate naive baselines |
| Anomalies | Occasional spikes (events?) visible in time series | Models should be robust to outliers; RMSE penalises spikes more than MAE |

---

## Selected Models

### Model 1: Stacked LSTM

**Architecture class:** Recurrent neural network with gated cells  
**Key papers:** Hochreiter & Schmidhuber (1997); Huang et al. (2017, cellular traffic); Vinayakumar et al. (2017)

**Justification:**
LSTMs are the established recurrent baseline for network traffic forecasting.
The gating mechanism (input, forget, output gates) allows the model to
selectively retain information over hundreds of time steps, making it
theoretically capable of capturing the daily and weekly periodicities
identified in the ACF analysis. Two stacked layers allow hierarchical
feature learning: the lower layer learns local patterns (hour-scale bumps)
while the upper layer can model the slower seasonal structure.

**Strengths:**
- Well-studied on this exact problem class
- Handles variable-length dependencies through hidden state
- Well-understood failure modes

**Limitations:**
- Sequential computation: cannot parallelise across time during training
- Vanishing gradient over very long sequences despite gating
- Hidden state is a bottleneck summarising the entire history

---

### Model 2: Temporal Convolutional Network (TCN)

**Architecture class:** Dilated causal convolutional network  
**Key papers:** Bai et al. (2018); van den Oord et al. (2016, WaveNet); Borovykh et al. (2017)

**Justification:**
Bai et al. (2018) demonstrated empirically that TCNs match or exceed LSTM
performance across a wide range of sequence modelling benchmarks.
The exponentially growing dilation schedule allows the receptive field to
cover hundreds of time steps with only a few layers — directly addressing
the daily (144-step) periodicity revealed by the ACF. Unlike LSTMs, TCNs
are fully parallelisable during training since convolutions across the
sequence dimension do not depend on hidden state from previous steps.
The causal structure (no future leakage) makes it appropriate for
real-time one-step-ahead forecasting.

**Strengths:**
- Parallelisable: faster training than LSTMs
- Controllable receptive field via dilation and kernel size
- Residual connections stabilise deep networks

**Limitations:**
- Fixed receptive field: cannot dynamically attend to distant timesteps
- Memory of very long-range patterns limited by depth / kernel size
- Less established on mobile traffic than LSTM

---

### Model 3: Encoder-only Transformer

**Architecture class:** Self-attention based sequence encoder  
**Key papers:** Vaswani et al. (2017); Zhou et al. (2021, Informer); Nie et al. (2023, PatchTST)

**Justification:**
Transformer-based architectures have achieved state-of-the-art results
on long-range time series forecasting benchmarks (Zhou et al. 2021;
Wu et al. 2021). Self-attention computes pairwise dependencies between
every pair of time steps in O(L²) operations, allowing the model to
directly attend to the same time-of-day from yesterday or the same
day of last week — something both LSTM and TCN handle only implicitly
through their sequential processing structure.
An encoder-only design (no decoder) is appropriate for one-step-ahead
prediction where no autoregressive decoding is required.
Pre-Layer Normalisation improves training stability on smaller datasets.

**Strengths:**
- Direct, distance-independent dependency modelling via attention
- Can attend simultaneously to multiple periodic lags (e.g., lag 144 AND lag 1008)
- Architecturally the most distinct from LSTM and TCN: different inductive bias

**Limitations:**
- O(L²) memory and compute w.r.t. sequence length
- Can underperform on small datasets without careful regularisation
- Less interpretable than LSTM gating patterns

---

## Why These Three?

The three models represent three fundamentally different inductive biases
for sequential modelling:

| Model | Mechanism | Parallelisable | Receptive field |
|---|---|---|---|
| LSTM | Hidden state (gated RNN) | No | Theoretically unbounded |
| TCN | Local dilated conv + residual | Yes | Fixed, by design |
| Transformer | Global self-attention | Yes (training) | Full input window |

Minor variations of the same architecture (e.g., LSTM vs GRU, or
one-layer vs two-layer LSTM) would not constitute a meaningful comparison.
These three differ in their core computational paradigm and therefore
produce genuinely different failure modes, which Section 4 will investigate.

In [3]:
# Optional: compute receptive field sizes for the TCN configs we plan to try

def tcn_receptive_field(kernel_size: int, num_blocks: int) -> int:
    """Receptive field of a TCN with exponential dilation (d=2^i)."""
    return 1 + 2 * (kernel_size - 1) * (2 ** num_blocks - 1)

print('TCN Receptive Fields:')
print(f'  kernel=3, 4 blocks: {tcn_receptive_field(3,4)} steps ({tcn_receptive_field(3,4)*10/60:.1f} hours)')
print(f'  kernel=5, 4 blocks: {tcn_receptive_field(5,4)} steps ({tcn_receptive_field(5,4)*10/60:.1f} hours)')
print(f'  kernel=7, 4 blocks: {tcn_receptive_field(7,4)} steps ({tcn_receptive_field(7,4)*10/60:.1f} hours)')
print(f'  kernel=5, 5 blocks: {tcn_receptive_field(5,5)} steps ({tcn_receptive_field(5,5)*10/60:.1f} hours)')
print()
print(f'  Daily cycle = 144 steps (24h)')
print(f'  Weekly cycle = 1008 steps (168h)')

TCN Receptive Fields:
  kernel=3, 4 blocks: 61 steps (10.2 hours)
  kernel=5, 4 blocks: 121 steps (20.2 hours)
  kernel=7, 4 blocks: 181 steps (30.2 hours)
  kernel=5, 5 blocks: 249 steps (41.5 hours)

  Daily cycle = 144 steps (24h)
  Weekly cycle = 1008 steps (168h)


In [5]:
# Parameter count comparison across candidate architectures
import sys
sys.path.insert(0, '../src')

import torch
from models import LSTMForecaster, TCNForecaster, TransformerForecaster

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

configs = [
    ('LSTM  (h=64,  L=2)', LSTMForecaster(1, hidden_size=64,  num_layers=2)),
    ('LSTM  (h=128, L=2)', LSTMForecaster(1, hidden_size=128, num_layers=2)),
    ('TCN   ([32,32,64,64], k=3)', TCNForecaster(1, [32,32,64,64], kernel_size=3)),
    ('TCN   ([64,64,64,64], k=5)', TCNForecaster(1, [64,64,64,64], kernel_size=5)),
    ('TF    (d=64,  h=4, L=2)',  TransformerForecaster(1, d_model=64,  nhead=4, num_layers=2, dim_feedfwd=128)),
    ('TF    (d=128, h=4, L=3)',  TransformerForecaster(1, d_model=128, nhead=4, num_layers=3, dim_feedfwd=256)),
]

print(f'{'Model':<40} {'Parameters':>12}')
print('-' * 54)
for name, model in configs:
    print(f'{name:<40} {count_params(model):>12,}')

Model                                      Parameters
------------------------------------------------------
LSTM  (h=64,  L=2)                             52,545
LSTM  (h=128, L=2)                            207,489
TCN   ([32,32,64,64], k=3)                     55,329
TCN   ([64,64,64,64], k=5)                    144,897
TF    (d=64,  h=4, L=2)                        69,185
TF    (d=128, h=4, L=3)                       406,017
